# Calibration across clinical sites (simulation)

Hold out each site in turn. Fit logistic regression on the other sites using 100 rounds of sample-weighted gradient aggregation. Training-site medians and scaling parameters are pooled in this simulation. Compare the original model with a locally fitted intercept using 30% of the held-out site's records. Evaluate on the disjoint 70% and weight sites equally. Lower Brier score is better. Generate four sites with 500 records each, different baseline disease rates and a shared feature effect.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'federated-cardiac-calibration-heavy-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.metrics import (\n    average_precision_score,\n    brier_score_loss,\n    roc_auc_score,\n)\nfrom sklearn.preprocessing import StandardScaler\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\ndef dataset(seed):\n    from scipy.special import expit\n\n    rng = np.random.default_rng(seed)\n    x = []\n    y = []\n    g = []\n    for s, bias in enumerate([-1.8, -0.6, 0.7, 1.6]):\n        features = rng.normal(size=(500, 9))\n        features[:, 0] += 0.3 * s\n        target = rng.random(500) < expit(features[:, 0] + 0.65 * features[:, 2] + bias)\n        x.append(features)\n        y.extend(target)\n        g.extend([s] * 500)\n    return np.vstack(x), np.array(y), np.array(g)\n\n\nfrom scipy.special import expit\nfrom sklearn.model_selection import train_test_split\n\n\ndef run(seed):\n    x, y, sites = dataset(seed)\n    measures = {"uncalibrated": [], "local_intercept": []}\n    saved = []\n    for site in np.unique(sites):\n        train = np.flatnonzero(sites != site)\n        local = np.flatnonzero(sites == site)\n        # Deterministic random calibration sample does not condition on local outcomes.\n        cal, test = train_test_split(local, test_size=0.7, random_state=seed)\n        med = np.nanmedian(x[train], axis=0)\n        med = np.nan_to_num(med)\n        z = np.where(np.isnan(x), med, x)\n        sc = StandardScaler().fit(z[train])\n        z = sc.transform(z)\n        z = np.c_[z, np.ones(len(z))]\n        w = np.zeros(z.shape[1])\n        clients = [np.flatnonzero(sites == key) for key in np.unique(sites[train])]\n        for _ in range(100):\n            grad = np.zeros_like(w)\n            for ids in clients:\n                grad += z[ids].T @ (expit(z[ids] @ w) - y[ids])\n            w -= 0.15 * (grad / len(train) + 0.012 * np.r_[w[:-1], 0])\n        offset = 0.0\n        for _ in range(30):\n            p = expit(z[cal] @ w + offset)\n            offset -= (np.sum(p - y[cal]) + offset) / (np.sum(p * (1 - p)) + 1.0)\n        for name, adj in [("uncalibrated", 0.0), ("local_intercept", offset)]:\n            p = expit(z[test] @ w + adj)\n            measures[name].append(\n                [brier_score_loss(y[test], p), abs(p.mean() - y[test].mean())]\n            )\n        saved.append((str(site), med, sc, w, offset, cal, test))\n    save(saved, f"sites-{seed}")\n    return {\n        k: dict(zip(["brier", "absolute_prevalence_error"], np.mean(v, axis=0)))\n        for k, v in measures.items()\n    }\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())